基于规则的关系抽取

In [1]:
import re

# 示例：简单的规则匹配
text = "马云创立了阿里巴巴"
pattern = r"(.+?)创立了(.+?)"
match = re.search(pattern, text)

if match:
    print(f"创始人: {match.group(1)}, 公司: {match.group(2)}")

创始人: 马云, 公司: 阿


使用 spaCy 进行关系抽取

In [2]:
import spacy

# 加载英文模型
nlp = spacy.load("en_core_web_sm")

text = "Apple was founded by Steve Jobs in 1976."
doc = nlp(text)

# 识别命名实体
for ent in doc.ents:
    print(ent.text, ent.label_)

Apple ORG
Steve Jobs PERSON
1976 DATE


使用 Hugging Face Transformers 进行文本分类

In [2]:
import os
from transformers import (AutoTokenizer,AutoModelForSequenceClassification,pipeline)

# ==================================================
# 1. 设置本地中文文本分类模型路径
# ==================================================
model_path = r"D:\11\NLP\data\jd-sentiment-local"

# ==================================================
# 2. 检查本地模型文件夹
# ==================================================
if not os.path.isdir(model_path):
    raise FileNotFoundError(f"找不到模型文件夹：\n{model_path}")

# ==================================================
# 3. 从本地加载分词器
# ==================================================
tokenizer = AutoTokenizer.from_pretrained(model_path,local_files_only=True)

# ==================================================
# 4. 从本地加载文本分类模型
# ==================================================
model = AutoModelForSequenceClassification.from_pretrained(model_path,local_files_only=True)

# ==================================================
# 5. 创建文本分类pipeline
# ==================================================
classifier = pipeline(task="text-classification",model=model,tokenizer=tokenizer,device=-1)
# ==================================================
# 6. 准备测试文本
# ==================================================

texts = [
    "这个手机运行速度很快，拍照效果也很好，我很满意。",
    "商品质量很差，刚使用一天就坏了。",
    "物流很快，包装也很完整。",
    "客服态度不好，问题一直没有得到解决。"
]

# ==================================================
# 7. 执行文本分类
# ==================================================
results = classifier(texts,truncation=True,max_length=512)

# ==================================================
# 8. 显示分类结果
# ==================================================
for text, result in zip(texts, results):
    original_label = result["label"]
    score = result["score"]
    # 把英文标签转换为中文
    if "negative" in original_label.lower():
        chinese_label = "负面"
    elif "positive" in original_label.lower():
        chinese_label = "正面"
    else:
        chinese_label = original_label
    print(f"文本：{text}")
    print(f"分类结果：{chinese_label}")
    print(f"原始标签：{original_label}")
    print(f"置信度：{score:.4f}")
    print("-" * 60)

Device set to use cpu


文本：这个手机运行速度很快，拍照效果也很好，我很满意。
分类结果：正面
原始标签：positive (stars 4 and 5)
置信度：0.9937
------------------------------------------------------------
文本：商品质量很差，刚使用一天就坏了。
分类结果：负面
原始标签：negative (stars 1, 2 and 3)
置信度：0.9847
------------------------------------------------------------
文本：物流很快，包装也很完整。
分类结果：正面
原始标签：positive (stars 4 and 5)
置信度：0.9918
------------------------------------------------------------
文本：客服态度不好，问题一直没有得到解决。
分类结果：负面
原始标签：negative (stars 1, 2 and 3)
置信度：0.9723
------------------------------------------------------------


数据准备（示例数据集）

In [1]:
# 示例数据集
data = [
    {
        "text": "比尔盖茨是微软的创始人",
        "relations": [{"head": "比尔盖茨", "tail": "微软", "type": "创始人"}]
    },
    {
        "text": "北京是中国的首都",
        "relations": [{"head": "北京", "tail": "中国", "type": "首都"}]
    }
]

# 查看数据
for item in data:
    print(f"文本: {item['text']}")
    print(f"关系: {item['relations']}\n")

文本: 比尔盖茨是微软的创始人
关系: [{'head': '比尔盖茨', 'tail': '微软', 'type': '创始人'}]

文本: 北京是中国的首都
关系: [{'head': '北京', 'tail': '中国', 'type': '首都'}]



特征工程（TF-IDF）

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 提取文本
texts = [d["text"] for d in data]

# TF-IDF 向量化
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

print("特征矩阵形状:", X.shape)
print("特征名称:", vectorizer.get_feature_names_out())

特征矩阵形状: (2, 2)
特征名称: ['北京是中国的首都' '比尔盖茨是微软的创始人']


模型训练（SVM）

In [3]:
from sklearn.svm import SVC

# 提取标签（简化示例，实际需要更复杂的标签处理）
y = [d["relations"][0]["type"] for d in data]

# 训练模型
model = SVC()
model.fit(X, y)

print("模型训练完成")
print("类别:", model.classes_)

模型训练完成
类别: ['创始人' '首都']


预测应用

In [5]:
# 测试新文本
test_text = "乔布斯创立了苹果公司"
test_vec = vectorizer.transform([test_text])
prediction = model.predict(test_vec)

print(f"预测关系: {prediction[0]}")

预测关系: 首都


补充 1：更完整的基于规则的关系抽取

In [6]:
import re

def extract_relations(text):
    relations = []

    # 定义多种关系模式
    patterns = [
        # 创始人关系
        (r"(.+?)创立了(.+?)", "创始人"),
        (r"(.+?)是(.+?)的创始人", "创始人"),
        # 首都关系
        (r"(.+?)是(.+?)的首都", "首都"),
        # 位于关系
        (r"(.+?)位于(.+?)", "位于"),
        # 任职关系
        (r"(.+?)担任(.+?)的(.+?)", "任职"),
    ]

    for pattern, rel_type in patterns:
        matches = re.findall(pattern, text)
        for match in matches:
            if len(match) == 2:
                relations.append({
                    "head": match[0],
                    "tail": match[1],
                    "type": rel_type
                })
            elif len(match) == 3:
                relations.append({
                    "head": match[0],
                    "tail": match[1],
                    "type": f"{rel_type}: {match[2]}"
                })

    return relations

# 测试
test_texts = [
    "马云创立了阿里巴巴",
    "北京是中国的首都",
    "张一鸣担任字节跳动的CEO"
]

for text in test_texts:
    result = extract_relations(text)
    print(f"文本: {text}")
    print(f"抽取关系: {result}\n")

文本: 马云创立了阿里巴巴
抽取关系: [{'head': '马云', 'tail': '阿', 'type': '创始人'}]

文本: 北京是中国的首都
抽取关系: [{'head': '北京', 'tail': '中国', 'type': '首都'}]

文本: 张一鸣担任字节跳动的CEO
抽取关系: [{'head': '张一鸣', 'tail': '字节跳动', 'type': '任职: C'}]



补充 2：使用 spaCy 进行关系抽取（规则+依存句法）

In [7]:
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_relations_spacy(text):
    doc = nlp(text)
    relations = []

    # 查找 "was founded by" 模式
    for token in doc:
        if token.lemma_ == "found" and token.dep_ == "ROOT":
            # 找到主语（通常是公司）
            subjects = [child for child in token.children if child.dep_ == "nsubjpass"]
            # 找到宾语（通常是创始人）
            objects = [child for child in token.children if child.dep_ == "agent"]

            for subj in subjects:
                for obj in objects:
                    relations.append({
                        "head": obj.text,  # 创始人
                        "tail": subj.text,  # 公司
                        "type": "founder"
                    })

    return relations

# 测试
text = "Apple was founded by Steve Jobs in 1976."
results = extract_relations_spacy(text)
print(results)

[{'head': 'by', 'tail': 'Apple', 'type': 'founder'}]


补充 3：使用预训练关系抽取模型（REBEL）

In [1]:
import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)


# ==================================================
# 1. 设置本地REBEL模型路径
# ==================================================

model_path = r"D:\11\NLP\data\rebel-large-local"


# ==================================================
# 2. 检查模型目录
# ==================================================

if not os.path.isdir(model_path):
    raise FileNotFoundError(
        f"找不到REBEL本地模型目录：\n{model_path}"
    )

required_files = [
    "config.json",
    "model.safetensors",
    "added_tokens.json",
    "merges.txt",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.json"
]

missing_files = []

for file_name in required_files:
    file_path = os.path.join(model_path, file_name)

    if not os.path.isfile(file_path):
        missing_files.append(file_name)

if missing_files:
    raise FileNotFoundError(
        "本地REBEL模型缺少以下文件：\n"
        + "\n".join(missing_files)
    )

print("本地REBEL模型文件检查成功。")


# ==================================================
# 3. 只从本地加载分词器和模型
# ==================================================
tokenizer = AutoTokenizer.from_pretrained(model_path,local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path,local_files_only=True)

# ==================================================
# 4. 设置运行设备
# ==================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print("模型运行设备：", device)

# ==================================================
# 5. 解析REBEL生成结果
# ==================================================
def parse_rebel_output(generated_text: str) -> list[dict]:
    """
    将REBEL生成的特殊格式文本解析为关系三元组。
    返回格式：
    [
        {
            "head": "Steve Jobs",
            "relation": "founder",
            "tail": "Apple Inc."
        }
    ]
    """
    triplets = []
    subject = ""
    relation = ""
    object_ = ""
    current_part = None

    # 删除普通的句子开始、结束和填充标记
    cleaned_text = (generated_text.replace("<s>", "").replace("</s>", "").replace("<pad>", "").strip())

    for token in cleaned_text.split():
        # 一个新三元组开始
        if token == "<triplet>":
            # 保存前一个完整三元组
            if subject and relation and object_:
                triplets.append({"head": subject.strip(),"relation": relation.strip(),"tail": object_.strip()})
            subject = ""
            relation = ""
            object_ = ""
            current_part = "subject"
        # REBEL中的<subj>后面实际上是宾语内容
        elif token == "<subj>":
            current_part = "object"
        # REBEL中的<obj>后面实际上是关系类型
        elif token == "<obj>":
            current_part = "relation"
        else:
            if current_part == "subject":
                subject += " " + token
            elif current_part == "object":
                object_ += " " + token
            elif current_part == "relation":
                relation += " " + token
    # 保存最后一个三元组
    if subject and relation and object_:
        triplets.append({"head": subject.strip(),"relation": relation.strip(), "tail": object_.strip() })
    return triplets

# ==================================================
# 6. 定义关系抽取函数
# ==================================================

def extract_relations_rebel(text: str) -> tuple[str, list[dict]]:
    """
    使用本地REBEL模型从英文文本中抽取关系。
    """
    if not isinstance(text, str) or not text.strip():
        raise ValueError("输入文本不能为空。")
    # 将文本转换为模型输入
    model_inputs = tokenizer( text,max_length=256,truncation=True, return_tensors="pt")

    # 将输入放到模型所在设备
    input_ids = model_inputs["input_ids"].to(device)
    attention_mask = model_inputs["attention_mask"].to(device)

    # 关闭梯度，提高推理效率
    with torch.no_grad():
        generated_tokens = model.generate(input_ids=input_ids,attention_mask=attention_mask, max_length=256, num_beams=3, num_return_sequences=1,length_penalty=0)
    # 必须保留<triplet>、<subj>、<obj>
    generated_text = tokenizer.decode(generated_tokens[0],skip_special_tokens=False)
    # 解析三元组
    triplets = parse_rebel_output(generated_text)
    return generated_text, triplets


# ==================================================
# 7. 测试
# ==================================================
text = "Steve Jobs was the co-founder of Apple Inc."
generated_text, relations = extract_relations_rebel(text)


# ==================================================
# 8. 输出结果
# ==================================================
print("\n原始文本：")
print(text)
print("\n模型生成的原始结果：")
print(generated_text)
print("\n抽取到的关系三元组：")
if not relations:
    print("没有识别到关系。")
else:
    for index, relation in enumerate(relations, start=1):
        print(f"\n关系 {index}")
        print(f"主体：{relation['head']}")
        print(f"关系：{relation['relation']}")
        print(f"客体：{relation['tail']}")

C:\Users\Administrator\.conda\envs\rl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


本地REBEL模型文件检查成功。
模型运行设备： cpu

原始文本：
Steve Jobs was the co-founder of Apple Inc.

模型生成的原始结果：
<s><triplet> Steve Jobs <subj> Apple Inc. <obj> employer <triplet> Apple Inc. <subj> Steve Jobs <obj> founded by</s>

抽取到的关系三元组：

关系 1
主体：Steve Jobs
关系：employer
客体：Apple Inc.

关系 2
主体：Apple Inc.
关系：founded by
客体：Steve Jobs
